# 05 Recommend: movies like this
Describe each movie by its tags (genres, decade, director, top 3 cast),
turn the tags into vectors, and find the closest movies with cosine similarity.

In [1]:
import csv
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

RAW = "../data/raw"
PROCESSED = "../data/processed"
TSV = dict(sep="\t", na_values="\\N", quoting=csv.QUOTE_NONE)

movies = pd.read_parquet(f"{PROCESSED}/movies.parquet")
print(movies.shape)

(49453, 34)


In [2]:
reader = pd.read_csv(
    f"{RAW}/title.principals.tsv.gz",
    **TSV,
    usecols=["tconst", "ordering", "nconst", "category"],
    dtype={"tconst": "string", "ordering": "int16",
           "nconst": "string", "category": "string"},
    chunksize=5_000_000,
)

pieces = []
for i, chunk in enumerate(reader):
    keep = chunk["tconst"].isin(movies["tconst"]) & chunk["category"].isin(["actor", "actress"])
    pieces.append(chunk[keep])
    print(f"chunk {i}: kept {keep.sum():,}")

principals = pd.concat(pieces, ignore_index=True)
del pieces
print(principals.shape)
principals["category"].value_counts()

chunk 0: kept 208,466
chunk 1: kept 15,842
chunk 2: kept 14,965
chunk 3: kept 13,667
chunk 4: kept 13,314
chunk 5: kept 14,216
chunk 6: kept 13,322
chunk 7: kept 17,676
chunk 8: kept 16,416
chunk 9: kept 16,736
chunk 10: kept 14,440
chunk 11: kept 10,943
chunk 12: kept 12,924
chunk 13: kept 11,198
chunk 14: kept 8,731
chunk 15: kept 7,824
chunk 16: kept 16,584
chunk 17: kept 18,551
chunk 18: kept 13,646
chunk 19: kept 10,830
chunk 20: kept 4,309
(474600, 4)


category
actor      311055
actress    163545
Name: count, dtype: int64[pyarrow]

In [3]:
principals = principals.sort_values(["tconst", "ordering"])
print("duplicates:", principals.duplicated(["tconst", "nconst"]).sum())
principals = principals.drop_duplicates(["tconst", "nconst"])

principals["cast_rank"] = principals.groupby("tconst").cumcount() + 1
cast = principals[principals["cast_rank"] <= 3].copy()
print(cast.shape, cast["tconst"].nunique(), "movies with cast")

duplicates: 12619
(141809, 5) 47592 movies with cast


In [4]:
names = pd.read_csv(
    f"{RAW}/name.basics.tsv.gz",
    **TSV,
    usecols=["nconst", "primaryName"],
    dtype={"nconst": "string", "primaryName": "string"},
)
names = names[names["nconst"].isin(cast["nconst"])]

cast = cast.merge(
    names.rename(columns={"primaryName": "name"}),
    on="nconst", how="left", validate="many_to_one",
)
del names, principals
print("missing names:", cast["name"].isna().sum())

cast.to_parquet(f"{PROCESSED}/cast.parquet", index=False)

dk = movies.loc[movies["title"] == "The Dark Knight", "tconst"].iloc[0]
cast[cast["tconst"] == dk]

missing names: 0


,tconst,ordering,nconst,category,cast_rank,name
62391,tt0468569,1,nm0000288,actor,1,Christian Bale
62392,tt0468569,2,nm0005132,actor,2,Heath Ledger
62393,tt0468569,3,nm0001173,actor,3,Aaron Eckhart


In [5]:
movies["decade"] = movies["year"] // 10 * 10
genres = movies[["tconst", "genre_list"]].explode("genre_list").dropna()

tags = pd.concat([
    pd.DataFrame({"tconst": genres["tconst"], "kind": "genre",
                  "tag": "genre:" + genres["genre_list"], "label": genres["genre_list"]}),
    pd.DataFrame({"tconst": movies["tconst"], "kind": "decade",
                  "tag": "decade:" + movies["decade"].astype("string"),
                  "label": movies["decade"].astype("string") + "s"}),
    pd.DataFrame({"tconst": movies["tconst"], "kind": "director",
                  "tag": "director:" + movies["director_id"], "label": movies["director"]}),
    pd.DataFrame({"tconst": cast["tconst"], "kind": "cast",
                  "tag": "cast:" + cast["nconst"], "label": cast["name"]}),
], ignore_index=True).dropna()

print(tags.shape)
print(tags["kind"].value_counts())
tags[tags["tconst"] == dk]

(355651, 4)
kind
cast        141809
genre       114951
decade       49453
director     49438
Name: count, dtype: int64


,tconst,kind,tag,label
1,tt0468569,genre,genre:Crime,Crime
2,tt0468569,genre,genre:Thriller,Thriller
114952,tt0468569,decade,decade:2000,2000s
164405,tt0468569,director,director:nm0634240,Christopher Nolan
276248,tt0468569,cast,cast:nm0000288,Christian Bale
276249,tt0468569,cast,cast:nm0005132,Heath Ledger
276250,tt0468569,cast,cast:nm0001173,Aaron Eckhart


In [6]:
tags["n_films"] = tags.groupby("tag")["tconst"].transform("size")
print("tag rows used by only one film:", (tags["n_films"] == 1).sum())

tags = tags[tags["n_films"] >= 2].copy()
print(tags.shape)
print(tags.groupby("kind")["tag"].nunique())
tags[tags["tconst"] == dk]

tag rows used by only one film: 49014
(306637, 5)
kind
cast        18054
decade         12
director     8174
genre          23
Name: tag, dtype: int64


,tconst,kind,tag,label,n_films
1,tt0468569,genre,genre:Crime,Crime,8259
2,tt0468569,genre,genre:Thriller,Thriller,7592
114952,tt0468569,decade,decade:2000,2000s,8489
164405,tt0468569,director,director:nm0634240,Christopher Nolan,13
276248,tt0468569,cast,cast:nm0000288,Christian Bale,40
276249,tt0468569,cast,cast:nm0005132,Heath Ledger,15
276250,tt0468569,cast,cast:nm0001173,Aaron Eckhart,39


In [7]:
row = pd.Index(movies["tconst"]).get_indexer(tags["tconst"])
col, tag_names = pd.factorize(tags["tag"])

def build_matrix(values):
    return csr_matrix((values, (row, col)), shape=(len(movies), len(tag_names)))

X = build_matrix(np.ones(len(tags)))
print(X.shape, f"{X.nnz:,} ones", f"{X.nnz / (X.shape[0] * X.shape[1]):.4%} filled")

(49453, 26263) 306,637 ones 0.0236% filled


In [8]:
def row_of(title):
    return movies.index[movies["title"] == title][0]

for a, b in [("The Dark Knight", "Batman Begins"),
             ("The Dark Knight", "The Dark Knight Rises"),
             ("The Dark Knight", "Toy Story")]:
    score = cosine_similarity(X[row_of(a)], X[row_of(b)])[0, 0]
    print(a, "vs", b, round(score, 3))

The Dark Knight vs Batman Begins 0.617
The Dark Knight vs The Dark Knight Rises 0.571
The Dark Knight vs Toy Story 0.0


In [9]:
def similar(title, n=5, min_votes=0):
    i = row_of(title)
    scores = cosine_similarity(X[i], X).ravel()
    out = movies[["title", "year", "director", "genres", "rating", "votes"]].assign(score=scores.round(3))
    out = out.drop(index=i)
    out = out[out["votes"] >= min_votes]
    return out.sort_values(["score", "votes"], ascending=False).head(n)

similar("The Dark Knight")

,title,year,director,genres,rating,votes,score
10386,Shade,2003,Damian Nieman,"Crime,Thriller",6.3,14003,0.655
19,Batman Begins,2005,Christopher Nolan,Crime,8.2,1753228,0.617
12,The Dark Knight Rises,2012,Christopher Nolan,"Crime,Thriller",8.4,2037297,0.571
8449,Fuck Me,2000,Virginie Despentes,"Crime,Drama,Thriller",4.5,19825,0.567
17520,The Horseman,2008,Steven Kastrissios,"Action,Crime,Thriller",6.5,5868,0.567


In [10]:
similar("Get Out")

,title,year,director,genres,rating,votes,score
5121,Unfriended: Dark Web,2018,Stephen Susco,"Horror,Mystery,Thriller",6.1,44068,0.707
15807,Hollow,2011,Michael Axelgaard,"Horror,Mystery,Thriller",4.7,6953,0.707
16368,Asylum of Fear,2018,Craig Deering,"Horror,Mystery,Thriller",3.5,6582,0.707
17787,Savageland,2015,Phil Guidry,"Horror,Mystery,Thriller",6.0,5713,0.707
25589,Lemon Tree Passage,2014,David Campbell,"Horror,Mystery,Thriller",3.9,3135,0.707


In [11]:
similar("Pride & Prejudice")

,title,year,director,genres,rating,votes,score
701,Atonement,2007,Joe Wright,"Drama,Mystery,Romance",7.8,335386,0.802
16031,Eyes Wide Open,2009,Haim Tabakman,"Drama,Romance",7.3,6798,0.655
24071,Dakota Skye,2008,John Humber,"Drama,Romance",6.4,3479,0.655
24899,Någon annanstans,2009,Hiwa Abbasi,"Drama,Romance",4.3,3279,0.655
28542,All My Life,2008,Maher Sabry,"Drama,Romance",4.5,2597,0.655


In [12]:
N = len(movies)
tags["idf"] = np.log(N / tags["n_films"])

(tags.drop_duplicates("tag").set_index("label")
     .loc[["Drama", "Horror", "2010s", "Christopher Nolan", "Heath Ledger"],
          ["kind", "n_films", "idf"]]
     .round(2))

,kind,n_films,idf
label,,,
Drama,genre,27543,0.59
Horror,genre,6693,2.0
2010s,decade,14561,1.22
Christopher Nolan,director,13,8.24
Heath Ledger,cast,15,8.1


In [13]:
KIND_WEIGHT = {"genre": 2.0, "decade": 1.0, "director": 1.0, "cast": 1.0}
tags["weight"] = tags["kind"].map(KIND_WEIGHT) * tags["idf"]

X = build_matrix(tags["weight"].to_numpy())

tags[tags["tconst"] == dk][["kind", "label", "n_films", "idf", "weight"]].round(2)

,kind,label,n_films,idf,weight
1,genre,Crime,8259,1.79,3.58
2,genre,Thriller,7592,1.87,3.75
114952,decade,2000s,8489,1.76,1.76
164405,director,Christopher Nolan,13,8.24,8.24
276248,cast,Christian Bale,40,7.12,7.12
276249,cast,Heath Ledger,15,8.1,8.1
276250,cast,Aaron Eckhart,39,7.15,7.15


In [14]:
similar("The Dark Knight")

,title,year,director,genres,rating,votes,score
12,The Dark Knight Rises,2012,Christopher Nolan,"Crime,Thriller",8.4,2037297,0.557
19,Batman Begins,2005,Christopher Nolan,Crime,8.2,1753228,0.527
23,The Prestige,2006,Christopher Nolan,"Drama,Mystery,Sci-Fi",8.5,1639716,0.441
3949,I'm Not There,2007,Todd Haynes,"Biography,Drama,Music",6.8,63897,0.406
2426,Following,1998,Christopher Nolan,"Crime,Mystery,Thriller",7.4,115158,0.393


In [16]:
similar("Get Out")

,title,year,director,genres,rating,votes,score
729,Nope,2022,Jordan Peele,"Horror,Mystery,Sci-Fi",6.8,325826,0.540
4397,The Perfection,2018,Richard Shepard,"Drama,Horror,Thriller",6.2,54762,0.405
546,Us,2019,Jordan Peele,"Horror,Mystery,Thriller",6.8,403144,0.382
5121,Unfriended: Dark Web,2018,Stephen Susco,"Horror,Mystery,Thriller",6.1,44068,0.375
15807,Hollow,2011,Michael Axelgaard,"Horror,Mystery,Thriller",4.7,6953,0.375


In [17]:
MIN_VOTES = 20_000
print("films that can be recommended:", (movies["votes"] >= MIN_VOTES).sum())

for t in ["The Dark Knight", "Toy Story", "Get Out", "Parasite", "Pride & Prejudice", "Oppenheimer"]:
    print("\n==", t)
    print(similar(t, min_votes=MIN_VOTES)[["title", "year", "director", "score"]].to_string())

films that can be recommended: 8412

== The Dark Knight
                      title  year           director  score
12    The Dark Knight Rises  2012  Christopher Nolan  0.557
19            Batman Begins  2005  Christopher Nolan  0.527
23             The Prestige  2006  Christopher Nolan  0.441
3949          I'm Not There  2007        Todd Haynes  0.406
2426              Following  1998  Christopher Nolan  0.393

== Toy Story
            title  year        director  score
227   Toy Story 2  1999   John Lasseter  0.765
2620  Toy Story 5  2026  Andrew Stanton  0.510
90    Toy Story 3  2010     Lee Unkrich  0.495
705   Toy Story 4  2019     Josh Cooley  0.479
332          Cars  2006   John Lasseter  0.447

== Get Out
                     title  year         director  score
729                   Nope  2022     Jordan Peele  0.540
4397        The Perfection  2018  Richard Shepard  0.405
546                     Us  2019     Jordan Peele  0.382
5121  Unfriended: Dark Web  2018    Stephen Susc

In [18]:
def explain(a, b):
    ta = tags[tags["tconst"] == movies.loc[row_of(a), "tconst"]]
    tb = tags[tags["tconst"] == movies.loc[row_of(b), "tconst"]]
    shared = ta[ta["tag"].isin(tb["tag"])]
    return shared[["kind", "label", "n_films", "weight"]].sort_values("weight", ascending=False).round(2)

explain("The Dark Knight", "I'm Not There")

,kind,label,n_films,weight
276249,cast,Heath Ledger,15,8.1
276248,cast,Christian Bale,40,7.12
114952,decade,2000s,8489,1.76


In [19]:
explain("Get Out", "Nope")

,kind,label,n_films,weight
164547,director,Jordan Peele,3,9.71
339650,cast,Daniel Kaluuya,4,9.42
366,genre,Mystery,4215,4.92
365,genre,Horror,6693,4.0


In [20]:
explain("Oppenheimer", "Son of God")

,kind,label,n_films,weight
181,genre,History,1944,6.47
179,genre,Biography,2623,5.87
180,genre,Drama,27543,1.17


In [21]:
tags.to_parquet(f"{PROCESSED}/tags.parquet", index=False)

- **Sprint 5: Recommender.** Built a "movies like this" recommender: each movie is a sparse vector of tags (genres, decade, director, top 3 cast), compared with cosine similarity.
- Version 1 (equal weights) ranked obscure films first, because thin profiles of 3-4 tags match too easily. IDF weighting (rare tags count more), double weight on genres and a 20,000-vote floor fixed it: Parasite now returns five Bong Joon Ho films.
- No answer key, so I sanity-checked by eye and built explain() to show shared tags. Limits: no plot data (Oppenheimer gets documentaries), and the vote floor adds popularity bias.
- there is a bug when the names are same and they may share different tags so explain may not compare the movie correctly, example below

In [22]:
similar("The Avengers")

,title,year,director,genres,rating,votes,score
86,Avengers: Age of Ultron,2015,Joss Whedon,"Action,Adventure,Sci-Fi",7.3,1029681,0.796
106,Captain America: Civil War,2016,Anthony Russo,"Action,Sci-Fi",7.8,956380,0.712
91,Captain America: The Winter Soldier,2014,Anthony Russo,"Action,Adventure,Sci-Fi",7.7,995878,0.528
32,Avengers: Endgame,2019,Anthony Russo,"Action,Adventure,Sci-Fi",8.4,1506243,0.514
45907,Molly,2017,Colinda Bongers,"Action,Sci-Fi",4.3,1144,0.413


In [27]:
explain("The Avengers","Molly")

,kind,label,n_films,weight


In [24]:
movies[movies["title"] == "Molly"]

,tconst,title,year,runtime,rating,votes,director_id,director,genres,genre_list,...,g_Mystery,g_News,g_Romance,g_Sci-Fi,g_Sport,g_Talk-Show,g_Thriller,g_War,g_Western,decade
24555,tt0143746,Molly,1999,102,5.7,3365,nm0241090,John Duigan,"Comedy,Drama,Romance","[Comedy, Drama, Romance]",...,0,0,1,0,0,0,0,0,0,1990
45907,tt6522982,Molly,2017,95,4.3,1144,nm4807084,Colinda Bongers,"Action,Sci-Fi","[Action, Sci-Fi]",...,0,0,0,1,0,0,0,0,0,2010
